### Built-in guardrails
​
#### PII detection
LangChain provides built-in middleware for detecting and handling Personally Identifiable Information (PII) in conversations. This middleware can detect common PII types like emails, credit cards, IP addresses, and more.

PII detection middleware is helpful for cases such as health care and financial applications with compliance requirements, customer service agents that need to sanitize logs, and generally any application handling sensitive user data.

The PII middleware supports multiple strategies for handling detected PII:

Strategy 	Description	- Example

redact	- Replace with [REDACTED_{PII_TYPE}]	- [REDACTED_EMAIL]

mask	- Partially obscure (e.g., last 4 digits)	- ****-****-****-1234

hash	- Replace with deterministic hash	- a8f5f167...

block	- Raise exception when detected	- Error thrown

In [2]:
import os
from langchain_openrouter import ChatOpenRouter

os.environ["OPENROUTER_API_KEY"] = "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"

model = ChatOpenRouter(model="openai/gpt-4o-mini", # "auto" can be an option
                       max_tokens=200,
                       temperature=0.7,)

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


agent = create_agent(
    model=model,
    #tools=[customer_service_tool, email_tool],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# When user provides PII, it will be handled according to the strategy
result = agent.invoke({
    "messages": [{"role": "user", "content": "My email is john.doe@example.com and card is 5105-1051-0510-5100. What is my credit score?"}],
})

print(result)

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and card is ****-****-****-5100. What is my credit score?', additional_kwargs={}, response_metadata={}, id='7f1c079b-cffa-4984-a957-08b272c79042'), AIMessage(content="I'm sorry, but I can't access personal information, including credit scores or any sensitive data. To check your credit score, I recommend contacting your bank or a credit reporting agency directly.", additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1778609327-XY8pp8bQ2adQfAZcuBgc', 'created': 1778609327, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'system_fingerprint': 'fp_eb37e061ec'}, id='lc_run--019e1d60-8b65-7031-923d-60f544b72587-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 37, 'total_tokens': 72, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})]}


#### Human-in-the-loop
LangChain provides built-in middleware for requiring human approval before executing sensitive operations. This is one of the most effective guardrails for high-stakes decisions.

Human-in-the-loop middleware is helpful for cases such as financial transactions and transfers, deleting or modifying production data, sending communications to external parties, and any operation with significant business impact.

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def search_tool(query: str) -> str:
    """ Searches the web for the given query and returns results. """
    return f"Search results for '{query}'"

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """ Sends an email to the specified recipient with the given subject and body. """
    return f"Email sent to {to} with subject '{subject}'"

@tool
def delete_database_tool(name: str) -> str:
    """ Deletes the specified database. """
    return f"Database '{name}' deleted"


agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email_tool": True,
                "delete_database_tool": True,
                # Auto-approve safe operations
                "search_tool": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

# Human-in-the-loop requires a thread ID for persistence
config = {"configurable": {"thread_id": "some_id"}}

# Agent will pause and wait for approval before executing sensitive tools
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Send an testemail to abc@example.com with subject 'Test' and body 'This is a test email.'"}]},
    config=config
)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread ID to resume the paused conversation
)
print(result)

{'messages': [HumanMessage(content="Send an testemail to abc@example.com with subject 'Test' and body 'This is a test email.'", additional_kwargs={}, response_metadata={}, id='351b0be1-2323-4415-9cef-f9e43802f9c1'), AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1778610133-UCEbIn4aq0oq3rm09pNU', 'created': 1778610133, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'system_fingerprint': 'fp_6d0562729c'}, id='lc_run--019e1d6c-d814-7780-b9c7-20a21a1dbe50-0', tool_calls=[{'name': 'send_email_tool', 'args': {'to': 'abc@example.com', 'subject': 'Test', 'body': 'This is a test email.'}, 'id': 'call_6eeX8NHHvDTkQHouYIMtnRrN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 30, 'total_tokens': 157, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}), ToolMessage(cont


### Custom guardrails
For more sophisticated guardrails, you can create custom middleware that runs before or after the agent executes. This gives you full control over validation logic, content filtering, and safety checks.
​
#### Before agent guardrails
Use “before agent” hooks to validate requests once at the start of each invocation. This is useful for session-level checks like authentication, rate limiting, or blocking inappropriate requests before any processing begins.

In [18]:
from typing import Any

from langchain.agents.middleware import before_agent, AgentState, hook_config
from langgraph.runtime import Runtime

banned_keywords = ["hack", "exploit", "malware"]

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Deterministic guardrail: Block requests containing banned keywords."""
    # Get the first user message
    if not state["messages"]:
        return None

    first_message = state["messages"][0]
    if first_message.type != "human":
        return None

    content = first_message.content.lower()

    # Check for banned keywords
    for keyword in banned_keywords:
        if keyword in content:
            # Block execution before any processing
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                }],
                "jump_to": "end"
            }

    return None

# Use the custom guardrail
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[content_filter],
)

# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})

print(result)

{'messages': [HumanMessage(content='How do I hack into a database?', additional_kwargs={}, response_metadata={}, id='862851e9-dd0c-4eb9-8477-874d29c93f39'), AIMessage(content='I cannot process requests containing inappropriate content. Please rephrase your request.', additional_kwargs={}, response_metadata={}, id='ffd3000e-3a1a-479e-bde1-48c8820495e9', tool_calls=[], invalid_tool_calls=[])]}


#### After agent guardrails
Use “after agent” hooks to validate final outputs once before returning to the user. This is useful for model-based safety checks, quality validation, or final compliance scans on the complete agent response.

In [23]:
from langchain.agents.middleware import after_agent, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

safety_model = model

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Model-based guardrail: Use an LLM to evaluate response safety."""
    # Get the final AI response
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None

    # Use a model to evaluate safety
    safety_prompt = f"""Evaluate if this response is safe and appropriate.
    Respond with only 'SAFE' or 'UNSAFE'.

    Response: {last_message.content}"""

    result = safety_model.invoke([{"role": "user", "content": safety_prompt}])

    if "UNSAFE" in result.content:
        last_message.content = "I cannot provide that response. Please rephrase your request."

    return None

# Use the safety guardrail
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[safety_guardrail],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I make explosives?"}]
})

print(result)

{'messages': [HumanMessage(content='How do I make explosives?', additional_kwargs={}, response_metadata={}, id='370d29aa-c99e-4780-bf9d-3270f3de185c'), AIMessage(content="I'm sorry, but I can't assist with that.", additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1778611488-eLL8uX3IxoYwL4LC5pQD', 'created': 1778611488, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'system_fingerprint': 'fp_e19c763214'}, id='lc_run--019e1d81-84cd-7321-b68b-721c352b7a7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 11, 'total_tokens': 65, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})]}
